In [1]:
import pandas as pd


In [2]:
df = pd.read_csv("IMDB_Dataset.csv", encoding = "latin-1")
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [3]:
df.shape

(50000, 2)

In [4]:
df.columns

Index(['review', 'sentiment'], dtype='object')

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   review     50000 non-null  object
 1   sentiment  50000 non-null  object
dtypes: object(2)
memory usage: 781.4+ KB


In [6]:
df.describe()

,review,sentiment
count,50000,50000
unique,49582,2
top,Loved today's show!!! It was a variety and not...,positive
freq,5,25000


In [7]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [8]:
df['sentiment'].value_counts()

sentiment
positive    25000
negative    25000
Name: count, dtype: int64

In [9]:
df.columns = df.columns.str.strip()
print(df.columns)
x = df["review"]
x.head()

Index(['review', 'sentiment'], dtype='object')


0    One of the other reviewers has mentioned that ...
1    A wonderful little production. <br /><br />The...
2    I thought this was a wonderful way to spend ti...
3    Basically there's a family where a little boy ...
4    Petter Mattei's "Love in the Time of Money" is...
Name: review, dtype: object

In [10]:
y = df["sentiment"]
y.head()

0    positive
1    positive
2    positive
3    negative
4    positive
Name: sentiment, dtype: object

In [11]:
from sklearn.preprocessing import LabelEncoder

In [12]:
encoder = LabelEncoder()
encoder

LabelEncoder()

In [13]:
y = encoder.fit_transform(y)
y[:11]

array([1, 1, 1, 0, 1, 1, 1, 0, 0, 1, 0])

In [14]:
print(encoder.classes_)

['negative' 'positive']


In [15]:
for i, review in enumerate(encoder.classes_):
    print(review, ":", i)

negative : 0
positive : 1


In [16]:
from sklearn.model_selection import train_test_split

In [17]:
x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size = 0.2,
    random_state = 42,
    stratify = y
)

In [18]:
print(x_train.shape)
print(x_test.shape)

print(y_train.shape)
print(y_test.shape)

(40000,)
(10000,)
(40000,)
(10000,)


In [19]:
import regex as re
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]','', text)
    return text

In [20]:
x_train = x_train.apply(clean_text)
x_test = x_test.apply(clean_text)

In [21]:
x_train.iloc[:5]

47808    i caught this little gem totally by accident b...
20154    i cant believe that i let myself into this mov...
43069    spoiler alert it just gets to me the nerve som...
19413    if theres one thing ive learnt from watching g...
13673    i remember when this was in theaters reviews s...
Name: review, dtype: object

In [22]:
from sklearn.feature_extraction.text import CountVectorizer

In [23]:
vectorizer = CountVectorizer()
vectorizer

,input,'content'
,encoding,'utf-8'
,decode_error,'strict'
,strip_accents,None
,lowercase,True
,preprocessor,None
,tokenizer,None
,stop_words,None
,token_pattern,'(?u)\\b\\w\\w+\\b'
,ngram_range,"(1, ...)"
,analyzer,'word'


In [24]:
x_train_bow = vectorizer.fit_transform(x_train)
vectorizer

,input,'content'
,encoding,'utf-8'
,decode_error,'strict'
,strip_accents,None
,lowercase,True
,preprocessor,None
,tokenizer,None
,stop_words,None
,token_pattern,'(?u)\\b\\w\\w+\\b'
,ngram_range,"(1, ...)"
,analyzer,'word'


In [25]:
x_test_bow = vectorizer.transform(x_test)

In [26]:
print(x_train_bow.shape)
print(x_test_bow.shape)

(40000, 154995)
(10000, 154995)


In [27]:
print(vectorizer.get_feature_names_out())

['aa' 'aaa' 'aaaaaaaargh' ... 'zzzzzzzzzzzzzzzzzzzzzzzzzzzzzzz'
 'zzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzz'
 'zzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzz']


In [28]:
print(x_train_bow[:5].toarray())

[[0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]]


In [29]:
from sklearn.naive_bayes import MultinomialNB

In [30]:
model = MultinomialNB()
model

,alpha,1.0
,force_alpha,True
,fit_prior,True
,class_prior,None


In [31]:
model.fit(x_train_bow,y_train)

,alpha,1.0
,force_alpha,True
,fit_prior,True
,class_prior,None


In [32]:
y_pred = model.predict(x_test_bow)
print(y_pred[:11])

[0 1 1 0 0 0 0 0 0 0 1]


In [33]:
result = pd.DataFrame({
    "Actual": y_test,
    "predicted":y_pred
})
print(result[:21])

    Actual  predicted
0        0          0
1        0          1
2        1          1
3        0          0
4        0          0
5        0          0
6        0          0
7        0          0
8        0          0
9        0          0
10       0          1
11       1          1
12       1          1
13       0          0
14       0          0
15       1          1
16       1          1
17       1          1
18       0          0
19       1          1
20       1          1


In [34]:
from sklearn.metrics import accuracy_score
accuracy = accuracy_score(y_test, y_pred)
print(accuracy)

0.8491


In [35]:
new_message = ["Movie is excellent  and very intresting."]
new_message = [clean_text(msg) for msg in new_message]
new_message

['movie is excellent  and very intresting']

In [52]:
new_message_vector = vectorizer.transform(new_message)
new_message_vector.toarray()

array([[0, 0, 0, ..., 0, 0, 0]], shape=(1, 154995))

In [53]:
prediction = model.predict(new_message_vector)
prediction

array([1])

In [54]:
if prediction[0] == 1:
    print("positive")
else:
    print("negative")

positive
